# Complemento GPU del E2: diagnósticos sobre los 50 checkpoints cerrados

**Exploratorio.** El test ya fue observado (E1 y evaluación cerrada E2). Nada de lo que salga aquí cambia modelos,
umbral (0,5) ni checkpoints; no hay optimizador ni entrenamiento. Todo se calibra en validación.

Qué produce, en `MyDrive/tesis_imo_e2_diagnosticos_v1/`:
- `clean`: train/val/test limpios con Dice, IoU, precisión, recall, HD95, ASSD, Brier, NLL, ECE fijo y adaptativo
  (15 bins), temperatura ajustada en validación y métricas recalibradas en test; latencia, memoria y FLOPs parciales;
  brecha train–val con el Dice de train en modo evaluación. 50 corridas.
- `robustness`: ruido gaussiano, desenfoque y brillo, 3 severidades cada uno, sobre test. 450 tareas.
- `interpret`: Integrated Gradients en 100 casos de validación para el top 3 por validación (TransUNet, U-Net, FCN),
  5 semillas; fidelidad (borrado/inserción contra orden aleatorio) y estabilidad entre semillas.
- Resumen: intervalos por bootstrap de grupos, curvas de robustez, ensembles de 5 semillas del top 3 con conjuntos de
  etiquetas y **predicción conforme por píxel con un umbral por clase (Mondrian) calibrada en validación**.

Necesitas en Drive: `tesis_imo_e2_arquitecturas_reales_v2/` (50 carpetas), `paquete_test_e2_cerrado.zip` y
`paquete_colab_e2_v2.zip` (o súbelos al panel de archivos). Entorno de ejecución con **GPU**, de preferencia la misma
en todas las sesiones: la carpeta de salida queda atada a la GPU de la primera corrida.

**Orden:** celdas 1 a 3 siempre (también tras reiniciar: `/content` se borra), luego el piloto, y después las etapas.
Cada etapa se puede volver a lanzar: lo terminado se verifica por firma y hash y se salta.


In [ ]:
# 1. Parámetros, Drive y utilidades
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, hashlib, zipfile, base64, time, shutil
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
RUNS = DRIVE/'tesis_imo_e2_arquitecturas_reales_v2'
OUTPUT = DRIVE/'tesis_imo_e2_diagnosticos_v1'      # resultados definitivos (reanudables)
PILOT = Path('/content/piloto_diagnosticos')       # piloto de tiempos: disco local, se descarta
LOGS = OUTPUT/'logs'
def first(*paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)
CLOSED_ZIP = first('/content/paquete_test_e2_cerrado.zip', DRIVE/'paquete_test_e2_cerrado.zip')
TRAINVAL_ZIP = first('/content/paquete_colab_e2_v2.zip', DRIVE/'paquete_colab_e2_v2.zip')
assert RUNS.is_dir(), f'Falta {RUNS}'
assert CLOSED_ZIP, 'Falta paquete_test_e2_cerrado.zip (raíz de Drive o panel de archivos)'
assert TRAINVAL_ZIP, 'Falta paquete_colab_e2_v2.zip (raíz de Drive o panel de archivos)'
LOGS.mkdir(parents=True, exist_ok=True)

def run(command, log=None, check=True):
    """Ejecuta mostrando la salida en vivo y, si se pide, copiándola a un log en Drive."""
    command = [str(c) for c in command]
    stream = open(log, 'a', encoding='utf-8') if log else None
    if stream: stream.write('\n$ ' + ' '.join(command) + '\n')
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy()) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
            if stream: stream.write(line); stream.flush()
        code = p.wait()
    if stream: stream.close()
    if check and code: raise RuntimeError(f'Comando fallido ({code}); revisa el error anterior: {command}')
    return code
def stamp(): return time.strftime('%Y%m%dT%H%M%S')
run(['nvidia-smi'])


In [ ]:
# 2. Escribir los archivos del complemento, armar el paquete y extraerlo (verifica todos los hashes)
EMBEDDED = {
"config/complementos_e2_20261120.json": [
"7df47a3c2703a4bf75171298d8a4d05459f0416c60bf1da27017024f99d51108",
"ewogICJzY2hlbWEiOiAxLAogICJkYXRlX2xvY2FsIjogIjIwMjYtMTEtMjAiLAogICJzdGF0dXMiOiAiUHJvdG9jb2xvIGRlIG51ZXZvcyBjb21wbGVtZW50b3MsIHBvc3RlcmlvciBhbCB0ZXN0IEUyOyBleHBsb3JhdG9yaW8sIG5vIHByZXJlZ2lzdHJvIHJldHJvYWN0aXZvIiwKICAiYXV0aG9yaXphdGlvbiI6ICJVc3VhcmlvIHNvbGljaXRhIGNvbXBsZXRhciB0b2RvcyBsb3MgcmVxdWlzaXRvcyBwYXJjaWFsZXMgZSBpbmNvbXBsZXRvcyIsCiAgInByZXNlcnZlX29yaWdpbmFsX3J1bnMiOiB0cnVlLAogICJ1c2VfdGVzdF9mb3Jfc2VsZWN0aW9uIjogZmFsc2UsCiAgInNlZWRzIjogWzExLCAyMywgMzcsIDUzLCA3MV0sCiAgInRvcDNfYnlfdmFsaWRhdGlvbiI6IFsidHJhbnN1bmV0XzJkIiwgInVuZXQiLCAiZmNuIl0sCiAgInRocmVzaG9sZCI6IDAuNSwKICAiYWxwaGEiOiAwLjA1LAogICJkZWx0YSI6IDAuMDIsCiAgImdhcF9tYXJnaW5fZXhwbG9yYXRvcnkiOiAwLjA1LAogICJib290c3RyYXBfZ3JvdXBzIjogNTAwMCwKICAiYmFzZWxpbmUiOiB7CiAgICAibW9kZWwiOiAiSGlzdEdyYWRpZW50Qm9vc3RpbmdDbGFzc2lmaWVyIiwKICAgICJmZWF0dXJlcyI6ICJSR0IsIEhTViwgcG9zaWNpb25lcyByZWxhdGl2YXMsIGdyYWRpZW50ZSB5IG1lZGlhcy9kZXN2aWFjaW9uZXMgbG9jYWxlcyBhIGVzY2FsYXMgMyB5IDkiLAogICAgInBpeGVsc19wZXJfdHJhaW5faW1hZ2UiOiAyNTYsCiAgICAic2FtcGxpbmciOiAidW5pZm9ybWUgc2luIHVzYXIgZXRpcXVldGE7IHRvZGFzIGxhcyBpbWFnZW5lcyB0cmFpbjsgc2VtaWxsYSBkZSBjb3JyaWRhIiwKICAgICJtYXhfbGVhZl9ub2Rlc19jYW5kaWRhdGVzIjogWzE1LCAzMSwgNjNdLAogICAgIm1heF9pdGVyIjogMTAwLAogICAgImxlYXJuaW5nX3JhdGUiOiAwLjEsCiAgICAibDJfcmVndWxhcml6YXRpb24iOiAxLjAsCiAgICAiZWFybHlfc3RvcHBpbmciOiBmYWxzZSwKICAgICJzZWxlY3Rpb24iOiAibWF5b3IgRGljZSBwb3IgZ3J1cG9zIGVuIHZhbGlkYWNpb24sIHByb21lZGlvIDUgc2VtaWxsYXM7IGVtcGF0ZSBtZW5vciBudW1lcm8gZGUgaG9qYXMiLAogICAgInRlc3QiOiAic29sbyBjb25maWd1cmFjaW9uIGVsZWdpZGEgcG9yIHZhbGlkYWNpb24sIDUgc2VtaWxsYXM7IG51ZXZhIGV2YWx1YWNpb24gZXhwbG9yYXRvcmlhIHNlcGFyYWRhIgogIH0sCiAgImFkdmVyc2FyaWFsIjogewogICAgImZlYXR1cmVzIjogIlJHQiBoaXN0b2dyYW1hcywgbWVkaWFzL2Rlc3ZpYWNpb25lcywgZW50cm9waWEsIGNvbnRyYXN0ZSwgZ3JhZGllbnRlLCBnZW9tZXRyaWE7IHNpbiBtYXNjYXJhIiwKICAgICJ1bml0IjogInByb21lZGlvIGRlIGRlc2NyaXB0b3JlcyBwb3IgZ3J1cG8iLAogICAgIm1vZGVsIjogIlN0YW5kYXJkU2NhbGVyICsgTG9naXN0aWNSZWdyZXNzaW9uIEM9MSBjbGFzc193ZWlnaHQ9YmFsYW5jZWQiLAogICAgImZvbGRzIjogNSwKICAgICJwZXJtdXRhdGlvbnMiOiAyMDAsCiAgICAic2VlZCI6IDIwMjYxMTIwCiAgfSwKICAiY2FsaWJyYXRpb24iOiB7ImJpbnNfZml4ZWQiOiAxNSwgImJpbnNfZXF1YWxfbWFzcyI6IDE1LCAidGVtcGVyYXR1cmVfZml0IjogInZhbGlkYWNpb24gc29sYW1lbnRlIiwgInRlbXBlcmF0dXJlX2JvdW5kcyI6IFswLjA1LCAxMC4wXX0sCiAgInJvYnVzdG5lc3MiOiB7Im5vaXNlX3NpZ21hIjogWzAuMDMsIDAuMDYsIDAuMTJdLCAiZ2F1c3NpYW5fYmx1cl9zaWdtYSI6IFswLjUsIDEuMCwgMi4wXSwgImJyaWdodG5lc3NfZmFjdG9yIjogWzAuOCwgMC42LCAwLjRdfSwKICAidW5jZXJ0YWludHkiOiB7Im1ldGhvZCI6ICJlbnNlbWJsZSBkZSBjaW5jbyBzZW1pbGxhcyBwb3IgbW9kZWxvIHRvcDMiLCAibm9taW5hbF9zZXRfbWFzcyI6IDAuOSwgImNsYWltIjogImNvYmVydHVyYSBlbXBpcmljYSBkZSBjb25qdW50b3MgZGUgZXRpcXVldGFzLCBzaW4gZ2FyYW50aWEgY29uZm9ybWUifSwKICAiY29uZm9ybWFsIjogeyJhZGRlZCI6ICIyMDI2LTExLTIwLCBhbnRlcyBkZSBlamVjdXRhciBkaWFnbm9zdGljb3MgR1BVIiwgIm1ldGhvZCI6ICJzcGxpdCBjb25mb3JtYWwgcG9yIHBpeGVsIGNvbmRpY2lvbmFkbyBhIGxhIGNsYXNlIChNb25kcmlhbik6IHVuIHVtYnJhbCBwYXJhIHJvdHVsbyB5IG90cm8gcGFyYSBmb25kbzsgZWwgdW1icmFsIHVuaWNvIG1hcmdpbmFsIHNlIHJlcG9ydGEgY29tbyByZWZlcmVuY2lhIiwgInNjb3JlIjogIjEgLSBwKGNsYXNlIHZlcmRhZGVyYSksIHAgPSBtZWRpYSBkZSBsYXMgNSBzZW1pbGxhcyBzaW4gdGVtcGVyYXR1cmEiLCAiY2FsaWJyYXRpb24iOiAidmFsaWRhY2lvbjsgdGVzdCBzb2xvIHBhcmEgbWVkaXIgY29iZXJ0dXJhIiwgImFscGhhIjogMC4xMCwgInBpeGVsc19wZXJfZ3JvdXBfYW5kX2NsYXNzIjogMjAwMCwgInNlZWQiOiAyMDI2MTExNywgImNsYWltIjogImdhcmFudGlhIG1hcmdpbmFsIGFwcm94aW1hZGE6IHBpeGVsZXMgZGUgdW5hIGltYWdlbiBubyBpbnRlcmNhbWJpYWJsZXMgeSB2YWxpZGFjaW9uIHlhIHVzYWRhIHBhcmEgY2hlY2twb2ludHM7IHNlIHJlcG9ydGEgbGEgY29iZXJ0dXJhIHBvciBncnVwbyB5IHBvciB0ZXJjaWwgZGUgYXJlYSJ9LAogICJib3JkZXJfbWV0cmljcyI6IHsidW5pdHMiOiAicGl4ZWxlcyBkZWwgbGllbnpvIDI1NiIsICJlbXB0eV9vbmVfbWFzayI6ICJIRDk1IHkgQVNTRCBwZW5hbGl6YWRhcyBwb3IgZGlhZ29uYWwgZGVsIGxpZW56bzsgcmVwb3J0YXIgZnJlY3VlbmNpYSBzZXBhcmFkYSIsICJlbXB0eV9ib3RoIjogMC4wfSwKICAiaW50ZXJwcmV0YWJpbGl0eSI6IHsibWV0aG9kIjogIkludGVncmF0ZWQgR3JhZGllbnRzIiwgInN0ZXBzIjogMTYsICJiYXNlbGluZSI6ICJSR0IgbWVkaW8gZGUgdHJhaW4iLCAibW9kZWxzIjogInRvcDMgcG9yIHZhbGlkYWNpb24iLCAic2VlZHMiOiBbMTEsMjMsMzcsNTMsNzFdLCAiY2FzZXMiOiAxMDAsICJmaWRlbGl0eV9mcmFjdGlvbnMiOiBbMC4xLDAuMiwwLjNdLCAic2VsZWN0aW9uIjogIm11ZXN0cmEgZXN0cmF0aWZpY2FkYSBkZSB2YWxpZGFjaW9uIHBvciBhcmVhLCBhbnRlcyBkZSBtYXBhcyJ9LAogICJsaW1pdHMiOiBbIkV4cG9zaWNpb24gcHJldmlhIGFsIHRlc3QiLCAiVmFsaWRhY2lvbiB5YSB1dGlsaXphZGEgcGFyYSBjaGVja3BvaW50cyIsICJHcnVwb3MgYXByb3hpbWFuIGVzY2VuYXMiLCAiTm8gY2FtYmlhciB1bWJyYWwgbmkgbW9kZWxvcyBjZXJyYWRvcyB0cmFzIGVzdG9zIGRpYWdub3N0aWNvcyIsICJRNSB5IFE2IHBlcm1hbmVjZW4gb21pdGlkYXMiXQp9Cg=="
],
"scripts/metricas_diagnostico_e2.py": [
"caf06dc81ad8c62dbaadf235ae298e39318981621ea43d8aaebfa7239aa8425d",
"IiIiTcOpdHJpY2FzIG51bcOpcmljYXMgcGFyYSBkaWFnbsOzc3RpY29zIEUyOyBzaW4gZnJhbWV3b3JrIG5pIGFqdXN0ZSBjb24gdGVzdC4iIiINCmltcG9ydCBudW1weSBhcyBucA0KZnJvbSBzY2lweSBpbXBvcnQgbmRpbWFnZSBhcyBuZGkNCmZyb20gc2NpcHkuc3BlY2lhbCBpbXBvcnQgZXhwaXQsbG9naXQNCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IG1pbmltaXplX3NjYWxhcg0KZnJvbSBkYXRhc2V0X3JlYWR5IGltcG9ydCBtZXRyaWNzDQoNCmRlZiByZWxpYWJpbGl0eShwLHksYmlucz0xNSxhZGFwdGl2ZT1GYWxzZSk6DQogICAgcD1ucC5hc2FycmF5KHApLnJhdmVsKCk7eT1ucC5hc2FycmF5KHkpLnJhdmVsKCkNCiAgICBpZiBhZGFwdGl2ZToNCiAgICAgICAgZWRnZXM9bnAudW5pcXVlKG5wLnF1YW50aWxlKHAsbnAubGluc3BhY2UoMCwxLGJpbnMrMSkpKQ0KICAgICAgICBiPW5wLnNlYXJjaHNvcnRlZChlZGdlc1sxOi0xXSxwLHNpZGU9J3JpZ2h0JykNCiAgICBlbHNlOmI9bnAubWluaW11bSgocCpiaW5zKS5hc3R5cGUoaW50KSxiaW5zLTEpDQogICAgbj1ucC5iaW5jb3VudChiLG1pbmxlbmd0aD1iaW5zKTthPW5wLmJpbmNvdW50KGIsd2VpZ2h0cz1wLG1pbmxlbmd0aD1iaW5zKTt0PW5wLmJpbmNvdW50KGIsd2VpZ2h0cz15LG1pbmxlbmd0aD1iaW5zKQ0KICAgIHJldHVybiBmbG9hdChucC5hYnMoYS10KS5zdW0oKS9sZW4ocCkpDQoNCmRlZiBwcm9iYWJpbGl0eV9tZXRyaWNzKHAseSx2KToNCiAgICBwPW5wLmFzYXJyYXkocClbdl0uYXN0eXBlKCdmbG9hdDY0Jyk7dD1ucC5hc2FycmF5KHkpW3ZdLmFzdHlwZSgnZmxvYXQ2NCcpDQogICAgej1ucC5jbGlwKHAsMWUtNywxLTFlLTcpDQogICAgcmV0dXJuIHsnQnJpZXInOmZsb2F0KG5wLm1lYW4oKHAtdCkqKjIpKSwgJ05MTCc6ZmxvYXQoLW5wLm1lYW4odCpucC5sb2coeikrKDEtdCkqbnAubG9nMXAoLXopKSksDQogICAgICAgICAgICAnRUNFMTUnOnJlbGlhYmlsaXR5KHAsdCksICdFQ0VfYWRhcHRpdmUxNSc6cmVsaWFiaWxpdHkocCx0LGFkYXB0aXZlPVRydWUpfQ0KDQpkZWYgY2FsaWJyYXRpb25fYmlucyhwLHksdixiaW5zPTE1KToNCiAgICB6PW5wLmFzYXJyYXkocClbdl0uYXN0eXBlKCdmbG9hdDY0Jyk7dD1ucC5hc2FycmF5KHkpW3ZdO2I9bnAubWluaW11bSgoeipiaW5zKS5hc3R5cGUoaW50KSxiaW5zLTEpDQogICAgcmV0dXJuIG5wLnN0YWNrKFtucC5iaW5jb3VudChiLG1pbmxlbmd0aD1iaW5zKSxucC5iaW5jb3VudChiLHdlaWdodHM9eixtaW5sZW5ndGg9YmlucyksDQogICAgICAgICAgICAgICAgICAgICBucC5iaW5jb3VudChiLHdlaWdodHM9dCxtaW5sZW5ndGg9YmlucyldLGF4aXM9MSkvbGVuKHopDQoNCmRlZiBmaXRfdGVtcGVyYXR1cmUobG9naXRzLGxhYmVscyx3ZWlnaHRzKToNCiAgICB6PW5wLmFzYXJyYXkobG9naXRzLGR0eXBlPSdmbG9hdDY0Jyk7eT1ucC5hc2FycmF5KGxhYmVscyxkdHlwZT0nZmxvYXQ2NCcpO3c9bnAuYXNhcnJheSh3ZWlnaHRzLGR0eXBlPSdmbG9hdDY0Jyk7dz13L3cuc3VtKCkNCiAgICBkZWYgbmxsKGxvZ3QpOg0KICAgICAgICB6ej16L25wLmV4cChsb2d0KTtyZXR1cm4gZmxvYXQobnAuc3VtKHcqKG5wLmxvZ2FkZGV4cCgwLHp6KS15Knp6KSkpDQogICAgZml0PW1pbmltaXplX3NjYWxhcihubGwsYm91bmRzPShucC5sb2coLjA1KSxucC5sb2coMTApKSxtZXRob2Q9J2JvdW5kZWQnKQ0KICAgIGlmIG5vdCBmaXQuc3VjY2VzczpyYWlzZSBSdW50aW1lRXJyb3IoJ0ZhbGxvIHRlbXBlcmF0dXJlIHNjYWxpbmcnKQ0KICAgIHJldHVybiB7J3RlbXBlcmF0dXJlJzpmbG9hdChucC5leHAoZml0LngpKSwndmFsaWRhdGlvbl9OTExfYmVmb3JlJzpubGwoMCksJ3ZhbGlkYXRpb25fTkxMX2FmdGVyJzpmbG9hdChmaXQuZnVuKSwNCiAgICAgICAgICAgICdib3VuZHMnOlsuMDUsMTAuXSwgJ2F0X2JvdW5kJzpib29sKGFicyhmaXQueC1ucC5sb2coLjA1KSk8MWUtMyBvciBhYnMoZml0LngtbnAubG9nKDEwKSk8MWUtMyl9DQoNCmRlZiBib3JkZXJfbWV0cmljcyhwLHksdik6DQogICAgcD0obnAuYXNhcnJheShwKT49LjUpJnY7eT1ucC5hc2FycmF5KHksYm9vbCkmdg0KICAgIHRwPWludCgocCZ5KS5zdW0oKSk7ZnA9aW50KChwJn55JnYpLnN1bSgpKTtmbj1pbnQoKH5wJnkpLnN1bSgpKQ0KICAgIHJlc3VsdD17J3ByZWNpc2lvbic6dHAvKHRwK2ZwKSBpZiB0cCtmcCBlbHNlIGZsb2F0KG5vdCB5LmFueSgpKSwNCiAgICAgICAgICAgICdyZWNhbGwnOnRwLyh0cCtmbikgaWYgdHArZm4gZWxzZSBmbG9hdChub3QgcC5hbnkoKSksDQogICAgICAgICAgICAnVFAnOnRwLCdGUCc6ZnAsJ0ZOJzpmbiwnZW1wdHlfcHJlZGljdGlvbic6aW50KG5vdCBwLmFueSgpKX0NCiAgICBpZiBub3QgcC5hbnkoKSBvciBub3QgeS5hbnkoKToNCiAgICAgICAgZGlzdD1mbG9hdChucC5oeXBvdCgqeS5zaGFwZSkpIGlmIHAuYW55KCkhPXkuYW55KCkgZWxzZSAwLg0KICAgICAgICByZXR1cm4geyoqcmVzdWx0LCdIRDk1JzpkaXN0LCdBU1NEJzpkaXN0fQ0KICAgIGJwPXAmfm5kaS5iaW5hcnlfZXJvc2lvbihwKTtieT15Jn5uZGkuYmluYXJ5X2Vyb3Npb24oeSkNCiAgICBkMT1uZGkuZGlzdGFuY2VfdHJhbnNmb3JtX2VkdCh+YnkpW2JwXTtkMj1uZGkuZGlzdGFuY2VfdHJhbnNmb3JtX2VkdCh+YnApW2J5XQ0KICAgIHJldHVybiB7KipyZXN1bHQsJ0hEOTUnOmZsb2F0KG1heChucC5wZXJjZW50aWxlKGQxLDk1KSxucC5wZXJjZW50aWxlKGQyLDk1KSkpLA0KICAgICAgICAgICAgJ0FTU0QnOmZsb2F0KChkMS5zdW0oKStkMi5zdW0oKSkvKGxlbihkMSkrbGVuKGQyKSkpfQ0KDQpkZWYgZGlhZ25vc2UocCx5LHYsd2l0aF9ib3JkZXI9VHJ1ZSk6DQogICAgZD17KiptZXRyaWNzKHAseSx2KSwqKnByb2JhYmlsaXR5X21ldHJpY3MocCx5LHYpfQ0KICAgIGlmIHdpdGhfYm9yZGVyOmQudXBkYXRlKGJvcmRlcl9tZXRyaWNzKHAseSx2KSkNCiAgICByZXR1cm4gZA0KDQpkZWYgcGVydHVyYih4LHYsbm9ybSxraW5kLGxldmVsLHNlZWQpOg0KICAgICIiIlRyYW5zZm9ybWEgY29udGVuaWRvIFJHQiwgc2luIG1lemNsYXIgcGFkZGluZyBuaSBjYW1iaWFyIGFub3RhY2nDs24uIiIiDQogICAgbWVhbj1ucC5hc2FycmF5KG5vcm1bJ21lYW4nXSxkdHlwZT0nZmxvYXQzMicpO3N0ZD1ucC5hc2FycmF5KG5vcm1bJ3N0ZCddLGR0eXBlPSdmbG9hdDMyJykNCiAgICByZ2I9eC50cmFuc3Bvc2UoMSwyLDApKnN0ZCttZWFuO3l5LHh4PW5wLndoZXJlKHYpDQogICAgc2w9KHNsaWNlKHl5Lm1pbigpLHl5Lm1heCgpKzEpLHNsaWNlKHh4Lm1pbigpLHh4Lm1heCgpKzEpKQ0KICAgIHBhdGNoPXJnYltzbF0uY29weSgpDQogICAgaWYga2luZD09J25vaXNlJzpwYXRjaCs9bnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpLm5vcm1hbCgwLGxldmVsLHBhdGNoLnNoYXBlKS5hc3R5cGUoJ2Zsb2F0MzInKQ0KICAgIGVsaWYga2luZD09J2JsdXInOnBhdGNoPW5kaS5nYXVzc2lhbl9maWx0ZXIocGF0Y2gsc2lnbWE9KGxldmVsLGxldmVsLDApKQ0KICAgIGVsaWYga2luZD09J2JyaWdodG5lc3MnOnBhdGNoKj1sZXZlbA0KICAgIGVsc2U6cmFpc2UgVmFsdWVFcnJvcihraW5kKQ0KICAgIHJnYltzbF09bnAuY2xpcChwYXRjaCwwLDEpO291dD0ocmdiLW1lYW4pL3N0ZDtvdXRbfnZdPTANCiAgICByZXR1cm4gb3V0LnRyYW5zcG9zZSgyLDAsMSkuYXN0eXBlKCdmbG9hdDMyJykNCg0KZGVmIGFnZ3JlZ2F0ZShyb3dzLGtleXM9Tm9uZSk6DQogICAgaWYga2V5cyBpcyBOb25lOmtleXM9W2sgZm9yIGssdiBpbiByb3dzWzBdLml0ZW1zKCkgaWYgaXNpbnN0YW5jZSh2LChpbnQsZmxvYXQpKSBhbmQgayBub3QgaW4gWydzZWVkJ11dDQogICAgZ3M9c29ydGVkKHtyWydncm91cF9pZCddIGZvciByIGluIHJvd3N9KQ0KICAgIHJldHVybiB7azpmbG9hdChucC5tZWFuKFtucC5tZWFuKFtyW2tdIGZvciByIGluIHJvd3MgaWYgclsnZ3JvdXBfaWQnXT09Z10pIGZvciBnIGluIGdzXSkpIGZvciBrIGluIGtleXN9DQoNCmRlZiBsYWJlbF9zZXQocCx5LHYsbWFzcz0uOSk6DQogICAgIiIiQ29uanVudG8gYmluYXJpbyBkZSBtw61uaW1hIGNhcmRpbmFsaWRhZCBjb24gbWFzYSA+PSBvYmpldGl2bzsgZW1ww61yaWNvLCBubyBjb25mb3JtZS4iIiINCiAgICBwcD1wW3ZdO3l5PXlbdl07c2luZ2xlPShucC5tYXhpbXVtKHBwLDEtcHApPj1tYXNzKQ0KICAgIGNvcnJlY3Q9KChwcD49LjUpPT15eSkNCiAgICByZXR1cm4geydjb3ZlcmFnZSc6ZmxvYXQobnAubWVhbih+c2luZ2xlfGNvcnJlY3QpKSwnbWVhbl9zZXRfc2l6ZSc6ZmxvYXQobnAubWVhbihucC53aGVyZShzaW5nbGUsMSwyKSkpLA0KICAgICAgICAgICAgJ3NpbmdsZXRvbl9mcmFjdGlvbic6ZmxvYXQoc2luZ2xlLm1lYW4oKSl9DQoNCiMgLS0tIFByZWRpY2Npw7NuIGNvbmZvcm1lIHBvciBww614ZWwsIGNvbmRpY2lvbmFkYSBhIGxhIGNsYXNlIChNb25kcmlhbikgLS0tDQojIFB1bnRhamUgZGUgbm8gY29uZm9ybWlkYWQgcyA9IDEgLSBwKGNsYXNlIHZlcmRhZGVyYSkuIFVuIHVtYnJhbCBwb3IgY2xhc2U6IGNvbiB1bm8gc29sbywgZWwgZm9uZG8NCiMgKH45NyAlIGRlIGxvcyBww614ZWxlcyB2w6FsaWRvcywgY2FzaSB0b2RvcyBmw6FjaWxlcykgc2UgbGxldmEgbGEgZ2FyYW50w61hIHkgbG9zIHDDrXhlbGVzIGRlIHLDs3R1bG8gcXVlZGFuDQojIHBvciBkZWJham8gZGUgbGEgbm9taW5hbC4gR2FyYW50w61hIGFwcm94aW1hZGE6IGxvcyBww614ZWxlcyBkZSB1bmEgaW1hZ2VuIG5vIHNvbiBpbnRlcmNhbWJpYWJsZXMuDQoNCmRlZiBjb25mb3JtYWxfcXVhbnRpbGUoc2NvcmVzLGFscGhhKToNCiAgICAiIiJrLcOpc2ltbyBtZW5vciBwdW50YWplIGNvbiBrID0gY2VpbCgobisxKSgxLWFscGhhKSk7IGluZiBzaSBrID4gbiAobXVlc3RyYSBkZW1hc2lhZG8gcGVxdWXDsWEpLg0KICAgIEVzdGFkw61zdGljbyBkZSBvcmRlbiBleGFjdG86IG5wLnF1YW50aWxlKC4uLiwgJ2hpZ2hlcicpIGVuIGsvbiBwdWVkZSB0b21hciBlbCBrKzEuIiIiDQogICAgcz1ucC5hc2FycmF5KHNjb3JlcyxkdHlwZT0nZmxvYXQ2NCcpLnJhdmVsKCk7bj1sZW4ocykNCiAgICBpZiBuPT0wOnJhaXNlIFZhbHVlRXJyb3IoJ1NpbiBwdW50YWplcyBkZSBjYWxpYnJhY2nDs24nKQ0KICAgIGs9aW50KG5wLmNlaWwoKG4rMSkqKDEtYWxwaGEpLTFlLTEyKSkNCiAgICByZXR1cm4gZmxvYXQoJ2luZicpIGlmIGs+biBlbHNlIGZsb2F0KG5wLnBhcnRpdGlvbihzLGstMSlbay0xXSkNCg0KZGVmIGNvbmZvcm1hbF9zY29yZXNfYnlfY2xhc3MocCx5LHYsZ3JvdXAsc3RvcmUpOg0KICAgICIiIkFjdW11bGEgZW4gc3RvcmVbY2xhc2VdW2dydXBvXSBoYXN0YSBwZXJfZ3JvdXAgcHVudGFqZXMgcG9yIGdydXBvIHkgY2xhc2UgKG11ZXN0cmVvIHNpbiByZWVtcGxhem8pLiIiIg0KICAgIHA9bnAuYXNhcnJheShwLGR0eXBlPSdmbG9hdDMyJyk7eT1ucC5hc2FycmF5KHkpLmFzdHlwZShib29sKTt2PW5wLmFzYXJyYXkodixib29sKQ0KICAgIGZvciBjLHMgaW4gKCgxLCgxLXApW3kmdl0pLCgwLHBbfnkmdl0pKToNCiAgICAgICAgaWYgbGVuKHMpOnN0b3JlW2NdLnNldGRlZmF1bHQoZ3JvdXAsW10pLmFwcGVuZChzKQ0KICAgIHJldHVybiBzdG9yZQ0KDQpkZWYgY29uZm9ybWFsX2NhbGlicmF0ZShzdG9yZSxhbHBoYSxwZXJfZ3JvdXAsc2VlZCk6DQogICAgIiIic3RvcmVbY11bZ3J1cG9dIC0+IGxpc3RhcyBkZSBwdW50YWplcy4gQ2FkYSBncnVwbyBhcG9ydGEgY29tbyBtw6F4aW1vIHBlcl9ncm91cCBww614ZWxlcyBwb3IgY2xhc2UuIiIiDQogICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKTtjYWw9e30NCiAgICBmb3IgYyBpbiAoMCwxKToNCiAgICAgICAgY2h1bmtzPVtdDQogICAgICAgIGZvciBnIGluIHNvcnRlZChzdG9yZVtjXSk6DQogICAgICAgICAgICBzPW5wLmNvbmNhdGVuYXRlKHN0b3JlW2NdW2ddKTtjaHVua3MuYXBwZW5kKHJuZy5jaG9pY2UocyxtaW4ocGVyX2dyb3VwLGxlbihzKSkscmVwbGFjZT1GYWxzZSkgaWYgbGVuKHMpPnBlcl9ncm91cCBlbHNlIHMpDQogICAgICAgIGNhbFtjXT1ucC5jb25jYXRlbmF0ZShjaHVua3MpDQogICAgcT17J2JhY2tncm91bmQnOmNvbmZvcm1hbF9xdWFudGlsZShjYWxbMF0sYWxwaGEpLCdsYWJlbCc6Y29uZm9ybWFsX3F1YW50aWxlKGNhbFsxXSxhbHBoYSksDQogICAgICAgJ21hcmdpbmFsJzpjb25mb3JtYWxfcXVhbnRpbGUobnAuY29uY2F0ZW5hdGUoW2NhbFswXSxjYWxbMV1dKSxhbHBoYSl9DQogICAgcmV0dXJuIHEseydiYWNrZ3JvdW5kJzpsZW4oY2FsWzBdKSwnbGFiZWwnOmxlbihjYWxbMV0pfQ0KDQpkZWYgY29uZm9ybWFsX3NldHMocCx5LHYscV9sYWJlbCxxX2JhY2tncm91bmQpOg0KICAgICIiIkNvbmp1bnRvIHBvciBww614ZWwge2M6IDEtcF9jIDw9IHFfY30uIERldnVlbHZlIGNvbnRlb3MgZGUgY29iZXJ0dXJhIHkgdGFtYcOxbyBwb3IgY2xhc2UgdmVyZGFkZXJhLiIiIg0KICAgIHA9bnAuYXNhcnJheShwLGR0eXBlPSdmbG9hdDMyJylbdl07dD1ucC5hc2FycmF5KHkpLmFzdHlwZShib29sKVt2XQ0KICAgIGluX2xhYmVsPSgxLXApPD1xX2xhYmVsO2luX2JnPXA8PXFfYmFja2dyb3VuZA0KICAgIGNvdmVyZWQ9bnAud2hlcmUodCxpbl9sYWJlbCxpbl9iZyk7c2l6ZT1pbl9sYWJlbC5hc3R5cGUoaW50KStpbl9iZy5hc3R5cGUoaW50KQ0KICAgIG91dD17fQ0KICAgIGZvciBuYW1lLHNlbCBpbiAoKCdhbGwnLG5wLm9uZXNfbGlrZSh0KSksKCdsYWJlbCcsdCksKCdiYWNrZ3JvdW5kJyx+dCkpOg0KICAgICAgICBvdXRbbmFtZV09eydjb3ZlcmVkJzppbnQoY292ZXJlZFtzZWxdLnN1bSgpKSwncGl4ZWxzJzppbnQoc2VsLnN1bSgpKSwnc2l6ZV9zdW0nOmludChzaXplW3NlbF0uc3VtKCkpLA0KICAgICAgICAgICAgICAgICAgICdlbXB0eSc6aW50KChzaXplW3NlbF09PTApLnN1bSgpKSwnYm90aCc6aW50KChzaXplW3NlbF09PTIpLnN1bSgpKX0NCiAgICByZXR1cm4gb3V0DQo="
],
"scripts/diagnosticos_gpu_e2.py": [
"..."
],
"scripts/resumir_diagnosticos_gpu_e2.py": [
"..."
],
"scripts/empaquetar_complemento_gpu_e2.py": [
"e96063a01414a76535578ea908c8f135cd7768205646a45bf41875be807dff4e",
"IiIiQXJtYSBwYXF1ZXRlX2NvbXBsZW1lbnRvX2dwdV9lMi56aXAgYSBwYXJ0aXIgZGVsIFpJUCBkZSB0ZXN0IGNlcnJhZG8sIHNpbiB0b2NhciBlbCBwcm90b2NvbG8gY2VycmFkby4KCk5vIHJlZXNjcmliZSBjb25maWcvcHJvdG9jb2xvX3Rlc3RfZTJfY2VycmFkby5qc29uIG5pIGVqZWN1dGEgcHJlcGFyYXJfdGVzdF9lMi5weS4gSGFjZSBlc3RvOgoxLiBWZXJpZmljYSBlbCBTSEEyNTYgZGVsIFpJUCBjZXJyYWRvIChlbCBxdWUgZmlqYSBjb2xhYi9FdmFsdWFjaW9uX0ZpbmFsX0UyX0NvbGFiLmlweW5iKSwgc3UKICAgUEFDS0FHRV9NQU5JRkVTVC5qc29uIHkgY2FkYSBhcmNoaXZvIGN1YmllcnRvIHBvciBsb3MgaGFzaGVzIGRlbCBwcm90b2NvbG8gY2VycmFkby4KMi4gQ29waWEgYnl0ZSBhIGJ5dGUgdG9kYXMgbGFzIGVudHJhZGFzIGRlbCBaSVAgY2VycmFkby4KMy4gQWdyZWdhIGxhcyBpbcOhZ2VuZXMgeSBtw6FzY2FyYXMgYmluYXJpYXMgZGUgdHJhaW4vdmFsIGxpc3RhZGFzIGVuIGxvcyBDU1YgZGVsIHByb3BpbyBaSVAgY2VycmFkbywKICAgdmVyaWZpY2FkYXMgY29udHJhIGltYWdlX3NoYTI1NiB5IGJpbmFyeV9tYXNrX3NoYTI1Ni4gRnVlbnRlcywgZW4gZXN0ZSBvcmRlbjogZWwgWklQIGRlIGVudHJlbmFtaWVudG8KICAgcGFxdWV0ZV9jb2xhYl9lMl92Mi56aXAgKC0tdHJhaW52YWwtemlwKSwgdW5hIGNhcnBldGEgZGUgZGF0b3MgKC0taW1hZ2VzLXJvb3QgLyAtLW1hc2tzLWRpcikgbywgcGFyYSBsYXMKICAgbcOhc2NhcmFzLCBkZXJpdmFybGFzIGRlIFNlZ21lbnRhdGlvbkNsYXNzICgtLWRlcml2ZS1tYXNrczsgc29sbyBzZSBhY2VwdGFuIHNpIGVsIFBORyByZXN1bHRhbnRlIGNvaW5jaWRlIGJ5dGUKICAgYSBieXRlIGNvbiBlbCBoYXNoIGRlbCBDU1YpLgo0LiBBZ3JlZ2EgZWwgcHJvdG9jb2xvIGNvbXBsZW1lbnRhcmlvIHkgbG9zIHNjcmlwdHMgZGUgZGlhZ27Ds3N0aWNvIChkZXNkZSAtLXNvdXJjZSwgcG9yIGRlZmVjdG8gZXN0ZSBwcm95ZWN0bykuCiAgIE5pbmd1bm8gcHVlZGUgcGlzYXIgdW5hIGVudHJhZGEgZGVsIFpJUCBjZXJyYWRvIGNvbiBjb250ZW5pZG8gZGlzdGludG8uCjUuIEVzY3JpYmUgUEFDS0FHRV9NQU5JRkVTVF9DT01QTEVNRU5UTy5qc29uIChoYXNoIGRlIGNhZGEgZW50cmFkYSB5IHByb2NlZGVuY2lhKSwgcmVsZWUgZWwgWklQIHkgZGVqYSAuc2hhMjU2LgoKU29sbyB1c2EgbGEgYmlibGlvdGVjYSBlc3TDoW5kYXIsIHNhbHZvIC0tZGVyaXZlLW1hc2tzIChudW1weSB5IFBpbGxvdykuIENvcnJlIGlndWFsIGVuIFdpbmRvd3MgeSBlbiBDb2xhYi4KIiIiCmltcG9ydCBhcmdwYXJzZSxjc3YsaGFzaGxpYixpbyxqc29uLG9zLHN5cyx6aXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUFJFRklYPSd0ZXNpc19pbW9fcGFyYV9jb2RleC8nCkNMT1NFRF9aSVBfU0hBMjU2PSdjZThiNmVjYzQ4ZGY2MThlNTllZDRhNGRlODZjNjkzNzMxZTQ0MzQ4NTc4MzA5MjFmNDBlZWRkZDhlZjYwNzZjJwpUUkFJTlZBTF9aSVBfU0hBMjU2PSc5NTRmNGZlNTY2M2Q5ZmVkZDY3NmRjZWU0ZjFmN2ZiOTA4NmEyYTdmNWMxOTA3Y2U5MDJiNWRmNGMxMDljYjQ5JwpDT01QTEVNRU5UPVsnY29uZmlnL2NvbXBsZW1lbnRvc19lMl8yMDI2MTEyMC5qc29uJywnc2NyaXB0cy9tZXRyaWNhc19kaWFnbm9zdGljb19lMi5weScsJ3NjcmlwdHMvZGlhZ25vc3RpY29zX2dwdV9lMi5weScsCiAgICAgICAgICAgICdzY3JpcHRzL3Jlc3VtaXJfZGlhZ25vc3RpY29zX2dwdV9lMi5weScsJ3NjcmlwdHMvZW1wYXF1ZXRhcl9jb21wbGVtZW50b19ncHVfZTIucHknLAogICAgICAgICAgICAnc2NyaXB0cy9wcnVlYmFfY3B1X2RpYWdub3N0aWNvc19lMi5weScsJ3Rlc3RzL3Rlc3RfbWV0cmljYXNfZGlhZ25vc3RpY29fZTIucHknXQoKZGVmIHNoYV9ieXRlcyhkYXRhKTpyZXR1cm4gaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCkKZGVmIHNoYV9maWxlKHBhdGgpOgogICAgaD1oYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwncmInKSBhcyBmOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTpmLnJlYWQoMTw8MjApLGInJyk6aC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKZGVmIHJlYWRfY2xvc2VkKHBhdGgsZXhwZWN0ZWQsbG9jYWxfdGVzdCk6CiAgICBhY3R1YWw9c2hhX2ZpbGUocGF0aCkKICAgIGlmIGFjdHVhbCE9ZXhwZWN0ZWQ6CiAgICAgICAgaWYgbm90IGxvY2FsX3Rlc3Q6cmFpc2UgUnVudGltZUVycm9yKGYnWklQIGNlcnJhZG8gZGlzdGludG86IHthY3R1YWx9ICE9IHtleHBlY3RlZH0nKQogICAgICAgIHByaW50KCdBVklTTyBwcnVlYmEgbG9jYWw6IFpJUCBjZXJyYWRvIHNpbiBlbCBoYXNoIG9maWNpYWwnLGFjdHVhbCxmbHVzaD1UcnVlKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUocGF0aCkgYXMgejoKICAgICAgICBpZiB6LnRlc3R6aXAoKSBpcyBub3QgTm9uZTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NSQyBpbnbDoWxpZG8gZW4gZWwgWklQIGNlcnJhZG8nKQogICAgICAgIGVudHJpZXM9e246ei5yZWFkKG4pIGZvciBuIGluIHoubmFtZWxpc3QoKSBpZiBub3Qgbi5lbmRzd2l0aCgnLycpfQogICAgbWFuaWZlc3Q9anNvbi5sb2FkcyhlbnRyaWVzWydQQUNLQUdFX01BTklGRVNULmpzb24nXSkKICAgIGZvciBuYW1lLGggaW4gbWFuaWZlc3QuaXRlbXMoKToKICAgICAgICBpZiBzaGFfYnl0ZXMoZW50cmllc1tuYW1lXSkhPWg6cmFpc2UgUnVudGltZUVycm9yKCdNYW5pZmllc3RvIGNlcnJhZG8gbm8gY29pbmNpZGU6ICcrbmFtZSkKICAgIHByb3RvY29sPWpzb24ubG9hZHMoZW50cmllc1tQUkVGSVgrJ2NvbmZpZy9wcm90b2NvbG9fdGVzdF9lMl9jZXJyYWRvLmpzb24nXSkKICAgIGZvciByZWwsaCBpbiBwcm90b2NvbFsnZmlsZXMnXS5pdGVtcygpOgogICAgICAgIGlmIFBSRUZJWCtyZWwgbm90IGluIGVudHJpZXMgb3Igc2hhX2J5dGVzKGVudHJpZXNbUFJFRklYK3JlbF0pIT1oOnJhaXNlIFJ1bnRpbWVFcnJvcignQXJjaGl2byBkZWwgcHJvdG9jb2xvIGNlcnJhZG8gYWx0ZXJhZG8gbyBhdXNlbnRlOiAnK3JlbCkKICAgIHJldHVybiBlbnRyaWVzLHByb3RvY29sLGFjdHVhbAoKZGVmIHBhbGV0dGUodGV4dCk6CiAgICBjb2xvcnM9W10KICAgIGZvciBsaW5lIGluIHRleHQuc3BsaXRsaW5lcygpOgogICAgICAgIGlmIGxpbmUuc3RyaXAoKSBhbmQgbm90IGxpbmUuc3RhcnRzd2l0aCgnIycpOmNvbG9ycy5hcHBlbmQodHVwbGUobWFwKGludCxsaW5lLnNwbGl0KCc6JylbMV0uc3BsaXQoJywnKSkpKQogICAgcmV0dXJuIGNvbG9ycwoKZGVmIGRlcml2ZV9tYXNrKGltYWdlc19yb290LHJvdyk6CiAgICAiIiJNaXNtYSBjb252ZXJzacOzbiBxdWUgcHJlcGFyYXJfYmFzZS5iaW5hcnlfbWFzayArIEltYWdlLnNhdmU7IHNvbG8gdmFsZSBzaSBlbCBoYXNoIGNvaW5jaWRlLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKICAgIHZvYz1QYXRoKGltYWdlc19yb290KS8nVk9DMjAxMic7Y29sb3JzPXBhbGV0dGUoKHZvYy8nbGFiZWxtYXAudHh0JykucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOC1zaWcnKSkKICAgIHdpdGggSW1hZ2Uub3BlbihQYXRoKGltYWdlc19yb290KS9yb3dbJ29yaWdpbmFsX21hc2tfcGF0aCddKSBhcyBpbTpyZ2I9bnAuYXNhcnJheShpbS5jb252ZXJ0KCdSR0InKSkKICAgIHBhY2tlZD1yZ2JbOiw6LDBdLmFzdHlwZShucC51aW50MzIpKjY1NTM2K3JnYls6LDosMV0uYXN0eXBlKG5wLnVpbnQzMikqMjU2K3JnYls6LDosMl0KICAgIGFsbG93ZWQ9W3IqNjU1MzYrZyoyNTYrYiBmb3IgcixnLGIgaW4gY29sb3JzXQogICAgaWYgbm90IG5wLmlzaW4ocGFja2VkLGFsbG93ZWQpLmFsbCgpOnJhaXNlIFJ1bnRpbWVFcnJvcignQ29sb3IgZGVzY29ub2NpZG8gZW4gJytyb3dbJ29yaWdpbmFsX21hc2tfcGF0aCddKQogICAgYnVmPWlvLkJ5dGVzSU8oKTtJbWFnZS5mcm9tYXJyYXkoKHBhY2tlZCE9MCkuYXN0eXBlKG5wLnVpbnQ4KSkuc2F2ZShidWYsZm9ybWF0PSdQTkcnKTtyZXR1cm4gYnVmLmdldHZhbHVlKCkKCmRlZiB0cmFpbnZhbF9maWxlcyhlbnRyaWVzLGFyZ3MpOgogICAgdHY9Tm9uZQogICAgaWYgYXJncy50cmFpbnZhbF96aXA6CiAgICAgICAgYWN0dWFsPXNoYV9maWxlKGFyZ3MudHJhaW52YWxfemlwKQogICAgICAgIGlmIGFjdHVhbCE9YXJncy50cmFpbnZhbF96aXBfc2hhMjU2OnJhaXNlIFJ1bnRpbWVFcnJvcihmJ1pJUCBkZSBlbnRyZW5hbWllbnRvIGRpc3RpbnRvOiB7YWN0dWFsfScpCiAgICAgICAgdHY9emlwZmlsZS5aaXBGaWxlKGFyZ3MudHJhaW52YWxfemlwKTtuYW1lcz1zZXQodHYubmFtZWxpc3QoKSkKICAgIGFkZGVkPXt9O29yaWdpbj17J3ppcCc6MCwnY2FycGV0YSc6MCwnZGVyaXZhZGEnOjB9CiAgICB0cnk6CiAgICAgICAgZm9yIHNwbGl0IGluIFsndHJhaW4nLCd2YWwnXToKICAgICAgICAgICAgcm93cz1saXN0KGNzdi5EaWN0UmVhZGVyKGlvLlN0cmluZ0lPKGVudHJpZXNbUFJFRklYK2YncmVzdWx0YWRvcy9wcmVlbnRyZW5hbWllbnRvL3tzcGxpdH0uY3N2J10uZGVjb2RlKCd1dGYtOC1zaWcnKSkpKQogICAgICAgICAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgICAgICAgICBpdGVtcz1bKHJvd1snaW1hZ2VfcGF0aCddLHJvd1snaW1hZ2Vfc2hhMjU2J10sJ2ltYWdlJyksKFBSRUZJWCtyb3dbJ2JpbmFyeV9tYXNrX3BhdGgnXSxyb3dbJ2JpbmFyeV9tYXNrX3NoYTI1NiddLCdtYXNrJyldCiAgICAgICAgICAgICAgICBmb3IgbmFtZSxleHBlY3RlZCxraW5kIGluIGl0ZW1zOgogICAgICAgICAgICAgICAgICAgIGRhdGE9Tm9uZTtob3c9Tm9uZQogICAgICAgICAgICAgICAgICAgIGlmIHR2IGlzIG5vdCBOb25lIGFuZCBuYW1lIGluIG5hbWVzOmRhdGE9dHYucmVhZChuYW1lKTtob3c9J3ppcCcKICAgICAgICAgICAgICAgICAgICBlbGlmIGtpbmQ9PSdpbWFnZScgYW5kIGFyZ3MuaW1hZ2VzX3Jvb3QgYW5kIChQYXRoKGFyZ3MuaW1hZ2VzX3Jvb3QpL25hbWUpLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgICAgICAgICAgZGF0YT0oUGF0aChhcmdzLmltYWdlc19yb290KS9uYW1lKS5yZWFkX2J5dGVzKCk7aG93PSdjYXJwZXRhJwogICAgICAgICAgICAgICAgICAgIGVsaWYga2luZD09J21hc2snIGFuZCBhcmdzLm1hc2tzX2RpciBhbmQgKFBhdGgoYXJncy5tYXNrc19kaXIpL1BhdGgobmFtZSkubmFtZSkuaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgICAgICBkYXRhPShQYXRoKGFyZ3MubWFza3NfZGlyKS9QYXRoKG5hbWUpLm5hbWUpLnJlYWRfYnl0ZXMoKTtob3c9J2NhcnBldGEnCiAgICAgICAgICAgICAgICAgICAgZWxpZiBraW5kPT0nbWFzaycgYW5kIGFyZ3MuZGVyaXZlX21hc2tzIGFuZCBhcmdzLmltYWdlc19yb290OgogICAgICAgICAgICAgICAgICAgICAgICBkYXRhPWRlcml2ZV9tYXNrKGFyZ3MuaW1hZ2VzX3Jvb3Qscm93KTtob3c9J2Rlcml2YWRhJwogICAgICAgICAgICAgICAgICAgIGlmIGRhdGEgaXMgTm9uZTpyYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmJ05vIGVuY3VlbnRybyB7bmFtZX0gZW4gbmluZ3VuYSBmdWVudGUnKQogICAgICAgICAgICAgICAgICAgIGlmIHNoYV9ieXRlcyhkYXRhKSE9ZXhwZWN0ZWQ6cmFpc2UgUnVudGltZUVycm9yKGYnSGFzaCBkaXN0aW50byBhbCBDU1YgKHtob3d9KToge25hbWV9JykKICAgICAgICAgICAgICAgICAgICBhZGRlZFtuYW1lXT1kYXRhO29yaWdpbltob3ddKz0xCiAgICBmaW5hbGx5OgogICAgICAgIGlmIHR2IGlzIG5vdCBOb25lOnR2LmNsb3NlKCkKICAgIHJldHVybiBhZGRlZCxvcmlnaW4KCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBhcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tY2xvc2VkLXppcCcscmVxdWlyZWQ9VHJ1ZSx0eXBlPVBhdGgpO2FwLmFkZF9hcmd1bWVudCgnLS1jbG9zZWQtemlwLXNoYTI1NicsZGVmYXVsdD1DTE9TRURfWklQX1NIQTI1NikKICAgIGFwLmFkZF9hcmd1bWVudCgnLS10cmFpbnZhbC16aXAnLHR5cGU9UGF0aCk7YXAuYWRkX2FyZ3VtZW50KCctLXRyYWludmFsLXppcC1zaGEyNTYnLGRlZmF1bHQ9VFJBSU5WQUxfWklQX1NIQTI1NikKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1pbWFnZXMtcm9vdCcsdHlwZT1QYXRoLGhlbHA9J0NhcnBldGEgcXVlIGNvbnRpZW5lIFZPQzIwMTIvIChpbWFnZV9wYXRoIGRlbCBDU1YgZXMgcmVsYXRpdm8gYSBlbGxhKScpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tbWFza3MtZGlyJyx0eXBlPVBhdGgsaGVscD0nQ2FycGV0YSBjb24gbGFzIG3DoXNjYXJhcyBiaW5hcmlhcyBpbWFnZV9YWFhYLnBuZycpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tZGVyaXZlLW1hc2tzJyxhY3Rpb249J3N0b3JlX3RydWUnKTthcC5hZGRfYXJndW1lbnQoJy0tc291cmNlJyx0eXBlPVBhdGgsZGVmYXVsdD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSdQcm95ZWN0byBkZWwgcXVlIHNlIHRvbWFuIGxvcyBhcmNoaXZvcyBkZWwgY29tcGxlbWVudG8nKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLW91dHB1dCcscmVxdWlyZWQ9VHJ1ZSx0eXBlPVBhdGgpO2FwLmFkZF9hcmd1bWVudCgnLS1sb2NhbC10ZXN0JyxhY3Rpb249J3N0b3JlX3RydWUnLAogICAgICAgICAgICAgICAgICAgIGhlbHA9J0FjZXB0YSB1biBaSVAgY2VycmFkbyBzaW4gZWwgaGFzaCBvZmljaWFsOyBxdWVkYSBtYXJjYWRvIGVuIGVsIG1hbmlmaWVzdG8uIE51bmNhIHBhcmEgbGEgY29ycmlkYSByZWFsLicpCiAgICBhcmdzPWFwLnBhcnNlX2FyZ3MoYXJndikKICAgIGVudHJpZXMscHJvdG9jb2wsY2xvc2VkX3NoYT1yZWFkX2Nsb3NlZChhcmdzLmNsb3NlZF96aXAsYXJncy5jbG9zZWRfemlwX3NoYTI1NixhcmdzLmxvY2FsX3Rlc3QpCiAgICBhZGRlZCxvcmlnaW49dHJhaW52YWxfZmlsZXMoZW50cmllcyxhcmdzKQogICAgZm9yIHJlbCBpbiBDT01QTEVNRU5UOmFkZGVkW1BSRUZJWCtyZWxdPShhcmdzLnNvdXJjZS9yZWwpLnJlYWRfYnl0ZXMoKQogICAgZm9yIG5hbWUsZGF0YSBpbiBhZGRlZC5pdGVtcygpOgogICAgICAgIGlmIG5hbWUgaW4gZW50cmllcyBhbmQgZW50cmllc1tuYW1lXSE9ZGF0YTpyYWlzZSBSdW50aW1lRXJyb3IoJ0VsIGNvbXBsZW1lbnRvIHBpc2Fyw61hIHVuYSBlbnRyYWRhIGNlcnJhZGE6ICcrbmFtZSkKICAgIHBheWxvYWQ9eyoqZW50cmllcywqKmFkZGVkfQogICAgbWFuaWZlc3Q9eydzY2hlbWEnOjEsJ2Nsb3NlZF96aXBfc2hhMjU2JzpjbG9zZWRfc2hhLCdjbG9zZWRfemlwX29mZmljaWFsJzpjbG9zZWRfc2hhPT1DTE9TRURfWklQX1NIQTI1NiwKICAgICAgICAgICAgICAndHJhaW52YWxfb3JpZ2luJzpvcmlnaW4sJ3RyYWludmFsX3ppcF9zaGEyNTYnOnNoYV9maWxlKGFyZ3MudHJhaW52YWxfemlwKSBpZiBhcmdzLnRyYWludmFsX3ppcCBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgJ2Nsb3NlZF9lbnRyaWVzJzpsZW4oZW50cmllcyksJ2FkZGVkX2VudHJpZXMnOmxlbihbbiBmb3IgbiBpbiBhZGRlZCBpZiBuIG5vdCBpbiBlbnRyaWVzXSksCiAgICAgICAgICAgICAgJ2xvY2FsX3Rlc3QnOmJvb2woYXJncy5sb2NhbF90ZXN0KSwnZmlsZXMnOntuOnNoYV9ieXRlcyhkKSBmb3IgbixkIGluIHNvcnRlZChwYXlsb2FkLml0ZW1zKCkpfX0KICAgIGFyZ3Mub3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7dGVtcD1QYXRoKHN0cihhcmdzLm91dHB1dCkrJy50bXAnKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUodGVtcCwndycsY29tcHJlc3Npb249emlwZmlsZS5aSVBfREVGTEFURUQsY29tcHJlc3NsZXZlbD02KSBhcyB6OgogICAgICAgIGZvciBuYW1lLGRhdGEgaW4gc29ydGVkKHBheWxvYWQuaXRlbXMoKSk6ei53cml0ZXN0cihuYW1lLGRhdGEpCiAgICAgICAgei53cml0ZXN0cignUEFDS0FHRV9NQU5JRkVTVF9DT01QTEVNRU5UTy5qc29uJyxqc29uLmR1bXBzKG1hbmlmZXN0LGluZGVudD0xLGVuc3VyZV9hc2NpaT1GYWxzZSkpCiAgICBvcy5yZXBsYWNlKHRlbXAsYXJncy5vdXRwdXQpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmdzLm91dHB1dCkgYXMgejoKICAgICAgICBpZiB6LnRlc3R6aXAoKSBpcyBub3QgTm9uZTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NSQyBpbnbDoWxpZG8gYWwgcmVsZWVyJykKICAgICAgICBmb3IgbmFtZSxoIGluIG1hbmlmZXN0WydmaWxlcyddLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIHNoYV9ieXRlcyh6LnJlYWQobmFtZSkpIT1oOnJhaXNlIFJ1bnRpbWVFcnJvcignUmVsZWN0dXJhIGRpc3RpbnRhOiAnK25hbWUpCiAgICAgICAgZm9yIHJlbCxoIGluIHByb3RvY29sWydmaWxlcyddLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIHNoYV9ieXRlcyh6LnJlYWQoUFJFRklYK3JlbCkpIT1oOnJhaXNlIFJ1bnRpbWVFcnJvcignSGFzaCBjZXJyYWRvIG5vIHByZXNlcnZhZG86ICcrcmVsKQogICAgZGlnZXN0PXNoYV9maWxlKGFyZ3Mub3V0cHV0KQogICAgUGF0aChzdHIoYXJncy5vdXRwdXQpKycuc2hhMjU2Jykud3JpdGVfdGV4dChmJ3tkaWdlc3R9ICB7YXJncy5vdXRwdXQubmFtZX1cbicsZW5jb2Rpbmc9J3V0Zi04JykKICAgIHByaW50KGpzb24uZHVtcHMoeyd6aXAnOnN0cihhcmdzLm91dHB1dCksJ01pQic6cm91bmQoYXJncy5vdXRwdXQuc3RhdCgpLnN0X3NpemUvMioqMjAsMSksJ3NoYTI1Nic6ZGlnZXN0LAogICAgICAgICAgICAgICAgICAgICAgJ2Nsb3NlZF9lbnRyaWVzJzptYW5pZmVzdFsnY2xvc2VkX2VudHJpZXMnXSwnYWRkZWRfZW50cmllcyc6bWFuaWZlc3RbJ2FkZGVkX2VudHJpZXMnXSwKICAgICAgICAgICAgICAgICAgICAgICd0cmFpbnZhbF9vcmlnaW4nOm9yaWdpbiwnY2xvc2VkX3ppcF9vZmZpY2lhbCc6bWFuaWZlc3RbJ2Nsb3NlZF96aXBfb2ZmaWNpYWwnXSwKICAgICAgICAgICAgICAgICAgICAgICdwcm90b2NvbF9maWxlc19wcmVzZXJ2ZWQnOmxlbihwcm90b2NvbFsnZmlsZXMnXSl9LGluZGVudD0xLGVuc3VyZV9hc2NpaT1GYWxzZSksZmx1c2g9VHJ1ZSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo="
],
"scripts/prueba_cpu_diagnosticos_e2.py": [
"9dd10fdac953eb9501ac2b92b8ef3f601b8fd1c1d7d7030f1dec8619cd227a20",
"IiIiUHJ1ZWJhIGRlIGh1bW8gZW4gQ1BVIGRlbCBjb21wbGVtZW50byBHUFU6IG1pc21hcyBmdW5jaW9uZXMgZGUgZXRhcGEsIG1vZGVsb3MgU0lOIEVOVFJFTkFSLCBwb2NhcyBpbcOhZ2VuZXMuDQoNClZhbGlkYSBlbCBmbHVqbyAobGVjdHVyYSBkZSBkYXRvcywgbcOpdHJpY2FzLCBjYWxpYnJhY2nDs24sIHJvYnVzdGV6LCBJRywgTlBaLCBmaXJtYXMsIHJlYW51ZGFjacOzbiB5IHJlc3VtaWRvciksDQpubyBsb3MgbsO6bWVyb3M6IGNvbiBwZXNvcyBhbGVhdG9yaW9zIERpY2UsIGNhbGlicmFjacOzbiB5IGF0cmlidWNpb25lcyBubyBzaWduaWZpY2FuIG5hZGEuIE5vIHVzYSBjaGVja3BvaW50cywNCm5vIHZlcmlmaWNhIGVsIGVudG9ybm8gZGVsIHByb3RvY29sbyB5IGRlamEgVk0tVU5ldCBmdWVyYSAoZXhpZ2UgQ1VEQSB5IG1hbWJhLXNzbSkuDQoNClVzbywgZGVudHJvIGRlbCBwYXF1ZXRlIGV4dHJhw61kbyBvIGRlbCBwcm95ZWN0byBjb24gbG9zIGRhdG9zIGFsIGxhZG86DQogIHB5dGhvbiBzY3JpcHRzL3BydWViYV9jcHVfZGlhZ25vc3RpY29zX2UyLnB5IC0tb3V0cHV0IDxjYXJwZXRhX251ZXZhPiAtLWxpbWl0IDMgLS1pZy1zdGVwcyAyDQp5IGRlc3B1w6lzOg0KICBweXRob24gc2NyaXB0cy9yZXN1bWlyX2RpYWdub3N0aWNvc19ncHVfZTIucHkgLS1vdXRwdXQgPGNhcnBldGFfbnVldmE+DQoiIiINCmltcG9ydCBhcmdwYXJzZSxqc29uLHRpbWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KaW1wb3J0IHRvcmNoDQppbXBvcnQgZGlhZ25vc3RpY29zX2dwdV9lMiBhcyBEDQpmcm9tIGV4cGVyaW1lbnRvX2UyX2NvbGFiIGltcG9ydCBQUk9KRUNULGRpZ2VzdCxhdG9taWNfanNvbixlbnZpcm9ubWVudCxzZXR1cF9zZWVkDQpmcm9tIG1vZGVsb3NfZTIgaW1wb3J0IE1PREVMU19FMg0KDQpkZWYgbWFpbigpOg0KICAgIGFwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxyZXF1aXJlZD1UcnVlLHR5cGU9UGF0aCk7YXAuYWRkX2FyZ3VtZW50KCctLWxpbWl0Jyx0eXBlPWludCxkZWZhdWx0PTMpDQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWlnLXN0ZXBzJyx0eXBlPWludCxkZWZhdWx0PTIpO2FwLmFkZF9hcmd1bWVudCgnLS1tb2RlbHMnLG5hcmdzPScqJykNCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2VlZHMnLG5hcmdzPScqJyx0eXBlPWludCk7YXAuYWRkX2FyZ3VtZW50KCctLXN0YWdlcycsbmFyZ3M9JyonLGRlZmF1bHQ9WydjbGVhbicsJ3JvYnVzdG5lc3MnLCdpbnRlcnByZXQnXSkNCiAgICBhcmdzPWFwLnBhcnNlX2FyZ3MoKTtvdXQ9YXJncy5vdXRwdXQ7b3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQ0KICAgIEQuREVWSUNFPSdjcHUnO0QuTElNSVQ9YXJncy5saW1pdDtELlBST0ZJTEVfV0FSTVVQLEQuUFJPRklMRV9SRVBTPTEsMw0KICAgIHByb3RvY29sPUQucmVhZChQUk9KRUNULydjb25maWcvcHJvdG9jb2xvX3Rlc3RfZTJfY2VycmFkby5qc29uJyk7Y2ZnPUQucmVhZChQUk9KRUNULydjb25maWcvY29tcGxlbWVudG9zX2UyXzIwMjYxMTIwLmpzb24nKQ0KICAgIGNmZ1snaW50ZXJwcmV0YWJpbGl0eSddWydzdGVwcyddPWFyZ3MuaWdfc3RlcHMNCiAgICB0cnk6ZW52PWVudmlyb25tZW50KCkNCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ZW52PXsnZXJyb3InOmYne3R5cGUoZSkuX19uYW1lX199OiB7ZX0nfQ0KICAgIHNpZ25hdHVyZT1kaWdlc3QoeydzbW9rZV9jcHVfdW50cmFpbmVkJzpUcnVlLCdjb25maWcnOmNmZywnbGltaXQnOmFyZ3MubGltaXQsJ3NjcmlwdCc6RC5zaGEoUGF0aChELl9fZmlsZV9fKSksDQogICAgICAgICAgICAgICAgICAgICAgJ21ldHJpY3MnOkQuc2hhKFBST0pFQ1QvJ3NjcmlwdHMvbWV0cmljYXNfZGlhZ25vc3RpY29fZTIucHknKX0pDQogICAgbG9jaz1vdXQvJ1BST1RPQ09MT19ESUFHTk9TVElDT1MuanNvbicNCiAgICBpZiBsb2NrLmV4aXN0cygpIGFuZCBELnJlYWQobG9jaylbJ3NpZ25hdHVyZSddIT1zaWduYXR1cmU6cmFpc2UgUnVudGltZUVycm9yKCdDYXJwZXRhIGRlIG90cmEgcHJ1ZWJhOyB1c2EgdW5hIG51ZXZhJykNCiAgICBhdG9taWNfanNvbihsb2NrLHsnc2lnbmF0dXJlJzpzaWduYXR1cmUsJ2NvbmZpZyc6Y2ZnLCdlbnZpcm9ubWVudCc6ZW52LCdjbG9zZWRfcHJvdG9jb2wnOnByb3RvY29sLCdsaW1pdCc6YXJncy5saW1pdCwNCiAgICAgICAgICAgICAgICAgICAgICAnV0FSTklORyc6J1BSVUVCQSBERSBIVU1PIEVOIENQVSBDT04gTU9ERUxPUyBTSU4gRU5UUkVOQVI6IGxvcyBuw7ptZXJvcyBubyB0aWVuZW4gc2lnbmlmaWNhZG8nfSkNCiAgICBub3JtPUQucmVhZChQUk9KRUNULydyZXN1bHRhZG9zL3ByZWVudHJlbmFtaWVudG8vbm9ybWFsaXphdGlvbl90cmFpbi5qc29uJykNCiAgICBtb2RlbHM9YXJncy5tb2RlbHMgb3IgW20gZm9yIG0gaW4gTU9ERUxTX0UyIGlmIG0hPSd2bXVuZXQnXTtza2lwcGVkPXt9DQogICAgZm9yIG0gaW4gbGlzdChtb2RlbHMpOg0KICAgICAgICB0cnk6TU9ERUxTX0UyW21dKCkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOnNraXBwZWRbbV09Zid7dHlwZShlKS5fX25hbWVfX306IHtlfSc7bW9kZWxzLnJlbW92ZShtKTtwcmludCgnU2luIHByb2JhciBlbiBlc3RlIGVudG9ybm86JyxtLHNraXBwZWRbbV0sZmx1c2g9VHJ1ZSkNCiAgICBydW5zPVtyIGZvciByIGluIHByb3RvY29sWydydW5zJ10gaWYgclsnbW9kZWwnXSBpbiBtb2RlbHMgYW5kIChub3QgYXJncy5zZWVkcyBvciByWydzZWVkJ10gaW4gYXJncy5zZWVkcyldDQogICAgdGltZXM9e30NCiAgICBmb3Igc3RhZ2UgaW4gYXJncy5zdGFnZXM6DQogICAgICAgIHQwPXRpbWUucGVyZl9jb3VudGVyKCkNCiAgICAgICAgZm9yIHJ1biBpbiBydW5zOg0KICAgICAgICAgICAgaWYgc3RhZ2U9PSdpbnRlcnByZXQnIGFuZCBydW5bJ21vZGVsJ10gbm90IGluIGNmZ1sndG9wM19ieV92YWxpZGF0aW9uJ106Y29udGludWUNCiAgICAgICAgICAgIHNldHVwX3NlZWQocnVuWydzZWVkJ10pO3RvcmNoLnNldF9udW1fdGhyZWFkcyg0KQ0KICAgICAgICAgICAgbW9kZWw9TU9ERUxTX0UyW3J1blsnbW9kZWwnXV0oKTttb2RlbC5ldmFsKCkNCiAgICAgICAgICAgIEQuU1RBR0VTW3N0YWdlXShtb2RlbCxydW4sb3V0LHNpZ25hdHVyZSxjZmcsbm9ybSkNCiAgICAgICAgdGltZXNbc3RhZ2VdPXRpbWUucGVyZl9jb3VudGVyKCktdDA7cHJpbnQoJ0V0YXBhJyxzdGFnZSwnbGlzdGEgZW4nLHJvdW5kKHRpbWVzW3N0YWdlXSwxKSwncycsZmx1c2g9VHJ1ZSkNCiAgICBhdG9taWNfanNvbihvdXQvJ3RpZW1wb3NfcHJ1ZWJhX2NwdS5qc29uJyx7J2xpbWl0JzphcmdzLmxpbWl0LCdpZ19zdGVwcyc6YXJncy5pZ19zdGVwcywncnVucyc6bGVuKHJ1bnMpLCdzZWNvbmRzJzp0aW1lcywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnZGV2aWNlJzonY3B1JywnbW9kZWxzJzptb2RlbHMsJ3NraXBwZWRfbm90X2J1aWxkYWJsZSc6c2tpcHBlZH0pDQoNCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQ0K"
],
"tests/test_metricas_diagnostico_e2.py": [
"5b708e9f06e76537940fa9652502613661f5a9f843be91c1958298290eeb80c7",
"aW1wb3J0IHN5cyx1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCnN5cy5wYXRoLmluc2VydCgwLHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXS8nc2NyaXB0cycpKQpmcm9tIG1ldHJpY2FzX2RpYWdub3N0aWNvX2UyIGltcG9ydCAocmVsaWFiaWxpdHkscHJvYmFiaWxpdHlfbWV0cmljcyxmaXRfdGVtcGVyYXR1cmUscGVydHVyYixsYWJlbF9zZXQsYm9yZGVyX21ldHJpY3MsCiAgICBjb25mb3JtYWxfcXVhbnRpbGUsY29uZm9ybWFsX3Njb3Jlc19ieV9jbGFzcyxjb25mb3JtYWxfY2FsaWJyYXRlLGNvbmZvcm1hbF9zZXRzKQpjbGFzcyBEaWFnbm9zdGljcyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9wZXJmZWN0X2FuZF9wYWRkaW5nKHNlbGYpOgogICAgICAgIHA9bnAuYXJyYXkoW1swLiwxLl0sWy45LC45XV0pO3k9bnAuYXJyYXkoW1swLDFdLFswLDBdXSk7dj1ucC5hcnJheShbWzEsMV0sWzAsMF1dLGJvb2wpCiAgICAgICAgcj1wcm9iYWJpbGl0eV9tZXRyaWNzKHAseSx2KTtzZWxmLmFzc2VydEVxdWFsKHJbJ0JyaWVyJ10sMCk7c2VsZi5hc3NlcnRFcXVhbChyWydFQ0UxNSddLDApCiAgICBkZWYgdGVzdF9hZGFwdGl2ZV9kb2VzX25vdF9zcGxpdF90aWVzX2J5X3BpeGVsX29yZGVyKHNlbGYpOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVsaWFiaWxpdHkobnAuZnVsbCgxMDAsLjUpLG5wLnJfW25wLnplcm9zKDUwKSxucC5vbmVzKDUwKV0sYWRhcHRpdmU9VHJ1ZSksMCkKICAgIGRlZiB0ZXN0X3RlbXBlcmF0dXJlX3JlZHVjZXNfbWlzY2FsaWJyYXRpb24oc2VsZik6CiAgICAgICAgej1ucC5mdWxsKDEwMCw0Lik7eT1ucC5yX1tucC5vbmVzKDc1KSxucC56ZXJvcygyNSldCiAgICAgICAgcj1maXRfdGVtcGVyYXR1cmUoeix5LG5wLm9uZXMoMTAwKSk7c2VsZi5hc3NlcnRHcmVhdGVyKHJbJ3RlbXBlcmF0dXJlJ10sMSkKICAgICAgICBzZWxmLmFzc2VydExlc3MoclsndmFsaWRhdGlvbl9OTExfYWZ0ZXInXSxyWyd2YWxpZGF0aW9uX05MTF9iZWZvcmUnXSkKICAgIGRlZiB0ZXN0X3BlcnR1cmJhdGlvbl9wcmVzZXJ2ZXNfcGFkZGluZ19hbmRfaXNfZGV0ZXJtaW5pc3RpYyhzZWxmKToKICAgICAgICB4PW5wLm9uZXMoKDMsOCw4KSxucC5mbG9hdDMyKSouNTt2PW5wLm9uZXMoKDgsOCksYm9vbCk7dlswXT1GYWxzZTt4WzosMF09MAogICAgICAgIG5vcm09eydtZWFuJzpbMCwwLDBdLCdzdGQnOlsxLDEsMV19CiAgICAgICAgYT1wZXJ0dXJiKHgsdixub3JtLCdub2lzZScsLjEsNDIpO2I9cGVydHVyYih4LHYsbm9ybSwnbm9pc2UnLC4xLDQyKQogICAgICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKGEsYik7c2VsZi5hc3NlcnRUcnVlKG5wLmFsbChhWzosMF09PTApKQogICAgZGVmIHRlc3RfbGFiZWxfc2V0c19hcmVfZW1waXJpY2FsKHNlbGYpOgogICAgICAgIHA9bnAuYXJyYXkoW1suOTUsLjUsLjA1XV0pO3k9bnAuYXJyYXkoW1sxLDEsMV1dKTt2PW5wLm9uZXNfbGlrZSh5LGJvb2wpCiAgICAgICAgcj1sYWJlbF9zZXQocCx5LHYpO3NlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoclsnY292ZXJhZ2UnXSwyLzMpO3NlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoclsnbWVhbl9zZXRfc2l6ZSddLDQvMykKICAgIGRlZiB0ZXN0X2JvcmRlcl9lbXB0eV9wZW5hbHR5KHNlbGYpOgogICAgICAgIHk9bnAub25lcygoMyw0KSxib29sKTtzZWxmLmFzc2VydEVxdWFsKGJvcmRlcl9tZXRyaWNzKG5wLnplcm9zX2xpa2UoeSkseSx5KVsnSEQ5NSddLDUpCiAgICBkZWYgdGVzdF9jb25mb3JtYWxfcXVhbnRpbGVfZmluaXRlX3NhbXBsZShzZWxmKToKICAgICAgICBzPW5wLmFyYW5nZSgxLDkpLzEwLjtzZWxmLmFzc2VydEVxdWFsKGNvbmZvcm1hbF9xdWFudGlsZShzLC4xKSxmbG9hdCgnaW5mJykpCiAgICAgICAgcz1ucC5hcmFuZ2UoMSwyMCkvMjAuO3NlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoY29uZm9ybWFsX3F1YW50aWxlKHMsLjEpLC45KQogICAgZGVmIHRlc3RfbW9uZHJpYW5fcHJvdGVjdHNfbWlub3JpdHlfY2xhc3Moc2VsZik6CiAgICAgICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZygwKTtzdG9yZT17MDp7fSwxOnt9fTt2PW5wLm9uZXMoKDY0LDY0KSxib29sKQogICAgICAgIGRlZiBzY2VuZSgpOgogICAgICAgICAgICB5PW5wLnplcm9zKCg2NCw2NCksbnAudWludDgpO3lbOjgsOjhdPTEKICAgICAgICAgICAgcD1ucC53aGVyZSh5PT0xLHJuZy51bmlmb3JtKC4yLDEsKDY0LDY0KSkscm5nLnVuaWZvcm0oMCwuMDUsKDY0LDY0KSkpCiAgICAgICAgICAgIHJldHVybiBwLHkKICAgICAgICBmb3IgZyBpbiByYW5nZSg0MCk6CiAgICAgICAgICAgIHAseT1zY2VuZSgpO2NvbmZvcm1hbF9zY29yZXNfYnlfY2xhc3MocCx5LHYsZidne2d9JyxzdG9yZSkKICAgICAgICBxLG49Y29uZm9ybWFsX2NhbGlicmF0ZShzdG9yZSwuMSwxMCoqNiwwKTtzZWxmLmFzc2VydEVxdWFsKG5bJ2xhYmVsJ10sNDAqNjQpCiAgICAgICAgY292PXsnbSc6WzAsMF0sJ21vbmQnOlswLDBdfQogICAgICAgIGZvciBfIGluIHJhbmdlKDQwKToKICAgICAgICAgICAgcCx5PXNjZW5lKCkKICAgICAgICAgICAgYT1jb25mb3JtYWxfc2V0cyhwLHksdixxWydsYWJlbCddLHFbJ2JhY2tncm91bmQnXSlbJ2xhYmVsJ107Yj1jb25mb3JtYWxfc2V0cyhwLHksdixxWydtYXJnaW5hbCddLHFbJ21hcmdpbmFsJ10pWydsYWJlbCddCiAgICAgICAgICAgIGNvdlsnbW9uZCddWzBdKz1hWydjb3ZlcmVkJ107Y292Wydtb25kJ11bMV0rPWFbJ3BpeGVscyddO2NvdlsnbSddWzBdKz1iWydjb3ZlcmVkJ107Y292WydtJ11bMV0rPWJbJ3BpeGVscyddCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKGNvdlsnbW9uZCddWzBdL2NvdlsnbW9uZCddWzFdLC44NykKICAgICAgICBzZWxmLmFzc2VydExlc3MoY292WydtJ11bMF0vY292WydtJ11bMV0sLjUpCiAgICBkZWYgdGVzdF9jb25mb3JtYWxfc2V0c19pZ25vcmVfcGFkZGluZyhzZWxmKToKICAgICAgICBwPW5wLmFycmF5KFtbLjksLjEsLjZdXSk7eT1ucC5hcnJheShbWzEsMCwwXV0pO3Y9bnAuYXJyYXkoW1sxLDEsMF1dLGJvb2wpCiAgICAgICAgcj1jb25mb3JtYWxfc2V0cyhwLHksdiwuNSwuNSk7c2VsZi5hc3NlcnRFcXVhbChyWydhbGwnXVsncGl4ZWxzJ10sMik7c2VsZi5hc3NlcnRFcXVhbChyWydhbGwnXVsnY292ZXJlZCddLDIpCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOnVuaXR0ZXN0Lm1haW4oKQo="
]
}

SOURCE = Path('/content/complemento_fuente/tesis_imo_para_codex')
for rel, (sha, b64) in EMBEDDED.items():
    data = base64.b64decode(b64)
    assert hashlib.sha256(data).hexdigest() == sha, rel
    (SOURCE/rel).parent.mkdir(parents=True, exist_ok=True); (SOURCE/rel).write_bytes(data)
print('Archivos del complemento:', {k: v[0][:12] for k, v in EMBEDDED.items()})
PACKAGE = Path('/content/paquete_complemento_gpu_e2.zip')
if not Path(str(PACKAGE) + '.sha256').exists():
    run([sys.executable, '-u', SOURCE/'scripts/empaquetar_complemento_gpu_e2.py', '--closed-zip', CLOSED_ZIP,
         '--trainval-zip', TRAINVAL_ZIP, '--output', PACKAGE], log=LOGS/f'paquete_{stamp()}.log')
def sha_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
package_sha = sha_file(PACKAGE)
assert package_sha == Path(str(PACKAGE) + '.sha256').read_text().split()[0], 'Paquete distinto de su .sha256'
BASE = Path('/content/imo_e2_complemento')
if BASE.exists(): shutil.rmtree(BASE)
with zipfile.ZipFile(PACKAGE) as z:
    for name in z.namelist(): assert (BASE/name).resolve().is_relative_to(BASE), 'Ruta inválida'
    z.extractall(BASE)
manifest = json.loads((BASE/'PACKAGE_MANIFEST_COMPLEMENTO.json').read_text())
assert manifest['closed_zip_official'] and not manifest['local_test'], 'El paquete no salió del ZIP cerrado oficial'
for name, expected in manifest['files'].items(): assert sha_file(BASE/name) == expected, name
for name, expected in json.loads((BASE/'PACKAGE_MANIFEST.json').read_text()).items(): assert sha_file(BASE/name) == expected, name
PROJECT = BASE/'tesis_imo_para_codex'; SCRIPTS = PROJECT/'scripts'
record = OUTPUT/'paquete_usado.json'
info = {'package_sha256': package_sha, 'closed_zip_sha256': manifest['closed_zip_sha256'],
        'trainval_zip_sha256': manifest['trainval_zip_sha256'], 'embedded': {k: v[0] for k, v in EMBEDDED.items()}}
if record.exists() and json.loads(record.read_text())['embedded'] != info['embedded']:
    raise RuntimeError('OUTPUT tiene resultados de otra versión de los scripts: usa otra carpeta de salida')
record.write_text(json.dumps(info, indent=1))
print('Paquete verificado:', package_sha, '|', manifest['added_entries'], 'archivos agregados al ZIP cerrado')


In [ ]:
# 3. Entorno aislado igual al del entrenamiento (Python 3.11, torch 2.5.1+cu121, Mamba). Repetir tras reiniciar.
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.6.9'])
ENV = Path('/content/imo-e2-env'); PY = str(ENV/'bin/python')
if not Path(PY).exists(): run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(ENV)])
def install(*items): run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *items])
install('setuptools==75.8.0')
install('torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121')
install('-r', str(PROJECT/'requirements-e2.txt'))
install('transformers==4.46.3')
install('--no-deps', 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
install('--no-deps', 'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
site = ENV/'lib/python3.11/site-packages'
dirs = [site/'torch/lib', *sorted((site/'nvidia').glob('*/lib'))]
paths = [str(p) for p in dirs if p.is_dir()] + os.environ.get('LD_LIBRARY_PATH', '').split(':')
os.environ['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(p for p in paths if p))
expected = json.loads((PROJECT/'config/protocolo_test_e2_cerrado.json').read_text())['training_environment']['packages']
check = (f"import importlib.metadata as m,json,torch,mamba_ssm,causal_conv1d; assert torch.cuda.is_available(); "
         f"e=json.loads({json.dumps(json.dumps(expected))}); a={{k:m.version(k) for k in e}}; "
         f"d={{k:(a[k],e[k]) for k in e if a[k]!=e[k]}}; assert not d, d; print('Entorno = protocolo;', torch.cuda.get_device_name(0))")
run([PY, '-u', '-c', check])
(OUTPUT/f'environment_{stamp()}.freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))
def stage(name, *extra, output=OUTPUT, log_dir=LOGS):
    run([PY, '-u', SCRIPTS/'diagnosticos_gpu_e2.py', '--runs', RUNS, '--output', output, '--stage', name, *extra],
        log=log_dir/f'{name}_{stamp()}.log')


## 4. Piloto de tiempos (antes de las 450 tareas de robustez)
Corre las tres etapas con la semilla 11 de cada red sobre pocas imágenes (`--limit`), en disco local, y extrapola al
total con los tiempos que registra cada tarea. **Es una estimación**: no incluye esperas de Drive ni cambios de GPU.
La verificación de los 50 checkpoints se paga una vez por cada lanzamiento de etapa y se mide aparte.


In [ ]:
# 4. Piloto: ~10-20 min según la GPU (también estimado). Escribe OUTPUT/PILOTO_TIEMPOS.json
N = 8
if PILOT.exists(): shutil.rmtree(PILOT)
PILOT.mkdir(parents=True)
for name, limit in [('clean', N), ('robustness', N), ('interpret', 2)]:
    stage(name, '--seeds', '11', '--limit', str(limit), output=PILOT)
cfg = json.loads((PROJECT/'config/complementos_e2_20261120.json').read_text())
FULL = {'train': 1015, 'val': 218, 'test': 218}; SEEDS = 5
logs = {json.loads(p.read_text())['stage']: json.loads(p.read_text()) for p in sorted((PILOT/'tiempos').glob('*.json'))}
load = {r['folder'].rsplit('_seed', 1)[0]: r['load_seconds'] for r in logs['clean']['runs']}
rows = []; total = {'clean': 0., 'robustness': 0., 'interpret': 0.}
for f in sorted(PILOT.glob('*_seed11_clean.json')):
    r = json.loads(f.read_text()); m = r['model']; s = r['seconds']
    per = {k: s[k] / r['splits'][k]['images'] for k in FULL}
    clean_s = SEEDS * (sum(per[k] * FULL[k] for k in FULL) + (s['total'] - sum(s[k] for k in FULL)) + load[m])
    rob = [json.loads(p.read_text()) for p in PILOT.glob(f'{m}_seed11_*_*.json') if '_clean' not in p.name and '_interpret' not in p.name]
    rob_s = SEEDS * (sum(x['seconds'] / x['images'] for x in rob) * FULL['test'] + load[m]) if rob else float('nan')
    ip = PILOT/f'{m}_seed11_interpret.json'
    int_s = SEEDS * (json.loads(ip.read_text())['seconds'] / json.loads(ip.read_text())['cases'] * cfg['interpretability']['cases'] + load[m]) if ip.exists() else 0.
    total['clean'] += clean_s; total['robustness'] += rob_s; total['interpret'] += int_s
    rows.append({'model': m, 'clean_min': clean_s / 60, 'robustness_min': rob_s / 60, 'interpret_min': int_s / 60})
verif = {k: v['verification_seconds'] for k, v in logs.items()}
for k in total: total[k] += verif[k]
estimate = {'ESTIMADO': 'extrapolado de un piloto con --limit, semilla 11; no es una medición de la corrida completa',
            'gpu': logs['clean']['device'], 'per_model': rows, 'verification_seconds': verif,
            'total_hours': {k: v / 3600 for k, v in total.items()}}
(OUTPUT/'PILOTO_TIEMPOS.json').write_text(json.dumps(estimate, indent=1, ensure_ascii=False))
for r in rows: print(f"{r['model']:15s} clean {r['clean_min']:7.1f} min | robustez {r['robustness_min']:7.1f} min | IG {r['interpret_min']:6.1f} min")
print('ESTIMADO total (h):', {k: round(v, 2) for k, v in estimate['total_hours'].items()})


## 5. Etapa `clean`
50 corridas × (1015 train + 218 val + 218 test). Guarda probabilidades float16 de val y test del top 3 (NPZ en Drive).
Si la sesión se corta: vuelve a correr las celdas 1 a 3 y esta misma; lo terminado se salta.


In [ ]:
# 5. Clean
stage('clean')


## 6. Etapa `robustness`
10 redes × 5 semillas × 3 perturbaciones × 3 severidades sobre las 218 imágenes de test. Revisa antes `PILOTO_TIEMPOS.json`.
Si la sesión se corta: vuelve a correr las celdas 1 a 3 y esta misma; lo terminado se salta.


In [ ]:
# 6. Robustez: 450 tareas. Para repartirla entre sesiones, pon aquí una lista de redes, p. ej. ['transunet_2d','vmunet']
MODELS = None
stage('robustness', *(['--models', *MODELS] if MODELS else []))


## 7. Etapa `interpret`
Top 3 por validación × 5 semillas × 100 casos de validación; IG de 16 pasos más 6 pasadas de fidelidad por caso.
Si la sesión se corta: vuelve a correr las celdas 1 a 3 y esta misma; lo terminado se salta.


In [ ]:
# 7. Explicabilidad
stage('interpret')


## 8. Resumen y 9. Exportación
El resumen se puede correr con etapas incompletas: marca `partial` y lista lo que falta.


In [ ]:
# 8. Resumen: verifica firmas y hashes (JSON y NPZ), calcula intervalos, ensembles y conformal en validación
run([PY, '-u', SCRIPTS/'resumir_diagnosticos_gpu_e2.py', '--output', OUTPUT], log=LOGS/f'resumen_{stamp()}.log')
summary = json.loads((OUTPUT/'RESUMEN_DIAGNOSTICOS.json').read_text())
print('Estado:', summary['status'], '| faltantes:', len(summary['missing']))
for e in summary['ensembles']:
    c = e['conformal']['mondrian_by_class']
    print(e['model'], 'cobertura conforme rótulo', round(c['label']['coverage'], 3), 'fondo', round(c['background']['coverage'], 3),
          '| marginal (un umbral) rótulo', round(e['conformal']['marginal_single_threshold_reference']['label']['coverage'], 3))


In [ ]:
# 9. Descargar los JSON (sin NPZ ni pesos; los NPZ quedan en Drive, verificados por hash en el resumen)
EXPORT = Path('/content/diagnosticos_gpu_e2_json.zip')
with zipfile.ZipFile(EXPORT, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and p.suffix in {'.json', '.log', '.txt'}: z.write(p, p.relative_to(OUTPUT))
print(EXPORT, round(EXPORT.stat().st_size / 2**20, 1), 'MiB')
from google.colab import files
files.download(str(EXPORT))
